# Bộ dữ liệu luận văn · m3av-maths

  1 lpm-videos            244 LPM videos (from thivntrung/dataset-video), checked, train/held-out split
  2 slidespeech-devtest   SlideSpeech dev + test videos and the related files (OCR, transcripts)
  3 slidespeech-train100  100 SlideSpeech train videos, streamed out of the 232 GB archive
  4 vn-test12             12 Vietnamese lecture videos from 5 channel types + SponsorBlock labels
  5 m3av-maths            M3AV Oxford Mathematics: slide OCR/LaTeX labels + the 51 videos


In [ ]:
import csv, glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, check=False):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr[-800:])
    return r

def probe(path):
    r = sh(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', str(path)])
    try:
        j = json.loads(r.stdout)
    except Exception:
        return None
    v = [s for s in j.get('streams', []) if s.get('codec_type') == 'video']
    a = [s for s in j.get('streams', []) if s.get('codec_type') == 'audio']
    return {'vcodec': v[0].get('codec_name', '') if v else '', 'width': int(v[0].get('width', 0)) if v else 0,
            'height': int(v[0].get('height', 0)) if v else 0, 'has_audio': bool(a),
            'acodec': a[0].get('codec_name', '') if a else '',
            'duration': round(float(j.get('format', {}).get('duration', 0) or 0), 2)}

def pyav_decodes(path, at=30.0):
    """True if PyAV (what stage 01 uses) can decode a frame of this file."""
    try:
        import av
        with av.open(str(path)) as c:
            s = c.streams.video[0]
            c.seek(int(at / float(s.time_base)), stream=s, backward=True, any_frame=False)
            for frame in c.decode(s):
                frame.to_ndarray(format='rgb24')
                return True
    except Exception as e:
        print('  PyAV lỗi', path.name, str(e)[:120])
    return False

def write_csv(path, rows):
    rows = list(rows)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def du_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 2**30

def install_ytdlp():
    sh([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]'])
    if not shutil.which('deno'):
        os.system('curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1')
        os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']
    print(sh([sys.executable, '-m', 'yt_dlp', '--version']).stdout.strip(), shutil.which('deno'))

BLOCKED_IN_A_ROW = [0]


def youtube(video_id, out, max_height=720, max_sec=None, tries=4, cookies=None, pause=(4, 10)):
    """mp4 with H.264 video (PyAV/OpenCV read it everywhere) and AAC audio; cut to max_sec.
    Uses deno explicitly for YouTube's JS challenge; pauses between videos; after 3 videos in a
    row are refused ("Sign in to confirm you're not a bot") it returns 'blocked' at once."""
    import random
    url = f'https://www.youtube.com/watch?v={video_id}'
    if BLOCKED_IN_A_ROW[0] >= 3:
        return {'status': 'blocked', 'reason': 'dừng: YouTube chặn 3 video liên tiếp'}
    time.sleep(random.uniform(*pause))
    deno = shutil.which('deno') or os.path.expanduser('~/.deno/bin/deno')
    js = ['--js-runtimes', f'deno:{deno}'] if os.path.exists(deno) else []
    formats = [f'bv*[height<={max_height}][vcodec^=avc1]+ba[ext=m4a]',
               f'bv*[height<={max_height}][ext=mp4]+ba[ext=m4a]', f'bv*[height<={max_height}]+ba/b']
    target = Path(out) / f'{video_id}.mp4'
    last = ''
    for attempt in range(tries):
        for f in Path(out).glob(f'{video_id}.*'):
            f.unlink()
        r = sh([sys.executable, '-m', 'yt_dlp', '-q', '--no-warnings', '--ignore-config', *js, '--no-playlist',
                '--socket-timeout', '30', '-f', formats[min(attempt, 2)],
                '--merge-output-format', 'mp4', '--retries', '10', '--fragment-retries', '10',
                *(([('--cookies-from-browser' if cookies[0] == 'browser' else '--cookies'), cookies[1]]) if cookies else []),
                '-o', str(target), url])
        if target.exists():
            info = probe(target)
            if info and info['height'] and info['has_audio']:
                if max_sec and info['duration'] > max_sec + 5:
                    cut = target.with_suffix('.cut.mp4')
                    sh(['ffmpeg', '-v', 'error', '-y', '-i', str(target), '-t', str(max_sec), '-c', 'copy', str(cut)], check=True)
                    original = info['duration']
                    cut.replace(target); info = probe(target); info['original_duration'] = original
                BLOCKED_IN_A_ROW[0] = 0
                return {'status': 'ok', **info}
            last = f'thiếu hình/tiếng: {info}'
        else:
            last = (r.stderr or '').strip().splitlines()[-1:] or ['không có file']
            last = last[0][:200]
            if 'Sign in' in last or 'not a bot' in last:
                BLOCKED_IN_A_ROW[0] += 1
                return {'status': 'blocked', 'reason': last}
        time.sleep(10 * (attempt + 1))
    return {'status': 'failed', 'reason': last}


In [ ]:
install_ytdlp()
sh([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
from huggingface_hub import hf_hub_download
OUT = Path('/kaggle/working/m3av-maths'); OUT.mkdir(parents=True, exist_ok=True)
TMP = Path('/kaggle/tmp/m3av'); TMP.mkdir(parents=True, exist_ok=True)
archive = hf_hub_download('CHHHH/M3AV_v1.0', 'dataset_v1.0_noaudio.tar.gz', repo_type='dataset', local_dir=str(TMP))
print('tải', round(Path(archive).stat().st_size / 2**30, 2), 'GB')
import tarfile
with tarfile.open(archive, 'r:gz') as t:
    names = [m.name for m in t if m.isfile()]
def is_mls(name):
    return any(part.upper().startswith('MLS') for part in Path(name).parts)
mls = [n for n in names if is_mls(n)]
print(len(names), 'file trong file nén ·', len(mls), 'file của Oxford Mathematics (MLS)')
print('ví dụ:', names[:5], mls[:8])
print('thư mục gốc:', sorted({n.split('/')[0] for n in names})[:10])
wanted = set(mls) | {n for n in names if n.endswith('.json') and n.count('/') <= 1}
with tarfile.open(archive, 'r:gz') as t:
    for member in t:
        if member.name in wanted:
            t.extract(member, OUT)
Path(archive).unlink()
for f in ('set_split.json', 'paper_link.json'):
    try:
        shutil.copy(hf_hub_download('CHHHH/M3AV_v1.0', f, repo_type='dataset', local_dir=str(TMP)), OUT / f)
    except Exception as e:
        print('không có', f, str(e)[:80])
# Tìm YouTube ID của từng video MLS trong file metadata
import re
video_dirs = sorted({next(p for p in Path(n).parts if p.upper().startswith('MLS')) for n in mls})
ids = {}
for f in OUT.rglob('*.ytbUrl'):
    m = re.search(r'(?:v=|youtu\.be/)([\w-]{11})', f.read_text(encoding='utf-8', errors='ignore'))
    if m:
        ids[m.group(1)] = f.name.split('.')[0]          # YouTube ID -> thư mục M3AV (MLS-xxxxx)
print(len(video_dirs), 'thư mục video MLS ·', len(ids), 'YouTube ID tìm thấy trong metadata')
(OUT / 'video').mkdir(exist_ok=True)
rows = []
for vid, where in sorted(ids.items()):
    result = youtube(vid, OUT / 'video')
    if result['status'] == 'ok':
        (OUT / 'video' / f'{vid}.mp4').rename(OUT / 'video' / f'{where}.mp4')
    print(where, vid, result.get('status'), result.get('duration'), result.get('reason', ''), flush=True)
    rows.append({'m3av_id': where, 'youtube_id': vid, 'file': f'video/{where}.mp4' if result['status'] == 'ok' else '', **{k: result.get(k, '') for k in ('status', 'duration', 'height', 'vcodec', 'reason')}})
    if result['status'] == 'blocked':
        break
if rows:
    write_csv(OUT / 'videos.csv', rows)
summary = {'mls_entries': len(mls), 'video_dirs': len(video_dirs), 'youtube_ids': len(ids),
           'videos_ok': sum(r['status'] == 'ok' for r in rows), 'size_gb': round(du_gb(OUT), 2)}
json.dump(summary, open(OUT / 'summary.json', 'w'), indent=1)
(OUT / 'README.md').write_text(f"""# M3AV · Oxford Mathematics (MLS)

The mathematics part of M3AV (Chen et al., ACL 2024, https://github.com/Jack-ZC8/M3AV-dataset),
licence CC BY-NC-SA 4.0: slide images with human OCR and LaTeX labels (printed and handwritten,
complex formulas) and word-level transcripts, plus the lecture videos from YouTube (`video/`).
Role in the thesis: scoring OCR on mathematical formulas (character error rate).
Summary: `summary.json`.
""", encoding='utf-8')
print(summary)
